# Example 9 — FBPINN: divide and conquer for complex problems

**FBPINN (Finite Basis PINN, Moseley et al.)** replaces one big network with many **small
networks on overlapping subdomains**, blended by smooth window functions:
$$u(x) = \sum_{j=1}^{J} w_j(x)\, N_j(\hat{x}_j),\qquad \sum_j w_j = 1\ \text{(partition of unity)},$$
where $\hat{x}_j$ is the input **rescaled to $[-1,1]$ within subdomain $j$**. It is the
neural version of finite elements: local basis functions with compact support — except each
basis function is a small learned network. (This is the architecture behind underPINN's
`FBPINNSolver`.)

**The killer feature — rescaling defeats spectral bias.** Take the high-frequency ODE
$$u'(x) = \omega\cos(\omega x),\quad u(0)=0,\qquad u_{exact}=\sin(\omega x),\quad \omega = 50\pi\ (k=25).$$
Example 6 showed a plain MLP cannot learn $k=25$ content — and here the *entire* solution
is $k=25$, so the plain PINN will fail totally. But chop $[0,1]$ into 30 patches: within one
patch the same wave completes **less than one oscillation** — in local coordinates it is
*low*-frequency, exactly what networks learn fastest. Same physics, easy problem.

**What you will see** (same training budget, comparable parameter counts ~9k):
- plain PINN: relative L2 error **≈ 1.0 forever** (outputs ~0 — it never sees the wave),
- FBPINN (30 nets of 2×16): error **≈ 0.01**.

Runs in ~1–2 minutes on Colab (CPU or GPU).

In [ ]:
# Cell 1 -- Problem + a look at the windows (the heart of FBPINN)
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

W = 50*np.pi                       # angular frequency (k = 25 oscillations in [0,1])
def u_exact(x): return torch.sin(W*x)
def f_rhs(x):   return W*torch.cos(W*x)

xg = torch.linspace(0, 1, 3001, device=device).reshape(-1, 1)   # evaluation grid

# --- the FBPINN geometry: J overlapping cosine^2 windows, normalised to sum to 1 ---
J, OVERLAP = 30, 1.9               # OVERLAP > 1 means neighbouring windows overlap
centers = torch.linspace(0, 1, J, device=device).reshape(1, J)
halfwidth = OVERLAP/(J-1)/2

def windows(x):                    # x: (N,1) -> weights (N,J), local coords (N,J)
    r = (x - centers)/halfwidth    # local coordinate, in [-1,1] inside the window
    inside = (r.abs() < 1).float()
    wt = torch.cos(np.pi*r/2)**2 * inside      # smooth compact bump
    w  = wt/(wt.sum(1, keepdim=True) + 1e-9)   # partition of unity
    return w, r

w, _ = windows(xg)
fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
ax[0].plot(xg.cpu(), u_exact(xg).cpu(), 'g', lw=0.8)
ax[0].set_ylabel('u'); ax[0].set_title(f'Target: sin({int(W/np.pi)}πx) — 25 oscillations')
ax[0].grid(alpha=.3)
ax[1].plot(xg.cpu(), w.cpu(), lw=0.9)
ax[1].plot(xg.cpu(), w.sum(1).cpu(), 'k--', lw=1.4, label='sum = 1 (partition of unity)')
ax[1].set_xlabel('x'); ax[1].set_ylabel('$w_j(x)$'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title(f'{J} overlapping windows — each patch sees <1 oscillation')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 2 -- The two contenders (comparable parameter counts)
def make_plain():
    torch.manual_seed(0)           # one big net: 1->64->64->64->1  (~8.5k params)
    return nn.Sequential(nn.Linear(1, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 1)).to(device)

class FBPINN(nn.Module):
    """J small nets (1->h->h->1) run in parallel via batched matmuls, blended by windows."""
    def __init__(self, J=30, h=16):
        super().__init__()
        torch.manual_seed(0)
        def P(*shape): return nn.Parameter(torch.randn(*shape)*0.5)
        self.W1=P(J,1,h); self.b1=P(J,1,h)
        self.W2=P(J,h,h); self.b2=P(J,1,h)
        self.W3=P(J,h,1); self.b3=P(J,1,1)
    def forward(self, x):                      # x: (N,1)
        w, r = windows(x)                      # weights (N,J), local coords (N,J)
        z = r.T.unsqueeze(2)                   # (J,N,1)  <- LOCAL coordinates, in [-1,1]
        z = torch.tanh(z@self.W1 + self.b1)    # all J nets at once
        z = torch.tanh(z@self.W2 + self.b2)
        z = (z@self.W3 + self.b3).squeeze(2).T # (N,J) local predictions
        return (w*z).sum(1, keepdim=True)      # blended global solution

def train(net, epochs=8000, tag=''):
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    hist = {'epoch': [], 'err': []}
    t0 = time.perf_counter()
    for e in range(epochs):
        opt.zero_grad()
        x = torch.rand(1024, 1, device=device).requires_grad_(True)
        u = net(x)
        ux = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
        loss = ((ux - f_rhs(x))**2).mean() + 100*net(torch.zeros(1, 1, device=device))[0, 0]**2
        loss.backward(); opt.step()
        if e % 200 == 0 or e == epochs-1:
            with torch.no_grad():
                err = torch.sqrt(torch.mean((net(xg)-u_exact(xg))**2)/torch.mean(u_exact(xg)**2)).item()
            hist['epoch'].append(e); hist['err'].append(err)
    if device.type == 'cuda': torch.cuda.synchronize()
    n = sum(p.numel() for p in net.parameters())
    print(f'{tag}: {n} params | {time.perf_counter()-t0:.0f} s | final rel L2 = {hist["err"][-1]:.4f}')
    return net, hist

net_plain, hist_plain = train(make_plain(), tag='plain PINN (1 big net) ')
net_fb,    hist_fb    = train(FBPINN().to(device),     tag='FBPINN (30 small nets) ')

In [ ]:
# Cell 3 -- Results
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

m = (xg.cpu().ravel() >= 0.4) & (xg.cpu().ravel() <= 0.6)   # zoom on 5 oscillations
xp = xg.cpu().ravel()
with torch.no_grad():
    up = net_plain(xg).cpu().ravel(); uf = net_fb(xg).cpu().ravel()
ax[0].plot(xp[m], u_exact(xg).cpu().ravel()[m], 'g', lw=1.8, label='exact')
ax[0].plot(xp[m], up[m], 'b--', lw=1.6, label=f'plain ({hist_plain["err"][-1]:.2f})')
ax[0].plot(xp[m], uf[m], 'r:', lw=2.0, label=f'FBPINN ({hist_fb["err"][-1]:.3f})')
ax[0].set_title('Plain never sees the wave; FBPINN overlays it')
ax[0].set_xlabel('x'); ax[0].set_ylabel('u'); ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)

ax[1].semilogy(hist_plain['epoch'], hist_plain['err'], 'b', label='plain (stuck at 1.0)')
ax[1].semilogy(hist_fb['epoch'],    hist_fb['err'],    'r', label='FBPINN')
ax[1].set_title('Same budget, comparable parameters (~9k each)')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('relative L2 error')
ax[1].legend(fontsize=9); ax[1].grid(alpha=.3, which='both')
plt.tight_layout(); plt.show()

## Takeaways

- **Rescaling, not fighting.** The plain PINN faces a $k=25$ target and spectral bias wins
  — its error never leaves 1.0. Each FBPINN patch sees the *same wave* as a
  less-than-one-oscillation, low-frequency function of its local coordinate. The global
  multiscale problem becomes 30 easy local ones.
- **Continuity is free.** The overlapping cosine² windows form a partition of unity, so the
  blended solution is smooth by construction — no interface penalty terms to balance
  (unlike XPINN/cPINN-style decompositions).
- **Locality conditions the optimization.** Each weight influences one patch only, so local
  corrections stay local — the gradient-interference mechanism behind Examples 7–8 is
  structurally confined.
- **This is how PINNs scale.** Large domains, long horizons, high frequencies: add patches,
  keep the networks tiny, train them (potentially in parallel). Time-slab decomposition —
  Example 8's cure — is FBPINN thinking applied to the $t$ axis. underPINN's
  `FBPINNSolver` and windowed transfer learning are two flavours of the same idea.
- **The kit's synthesis:** Fourier features (Ex. 6), RBA (Ex. 7) and causal weights (Ex. 8)
  fix the *loss or inputs*; FBPINN fixes the *architecture*. Real libraries ship all of them.

**Experiments to try:** raise `W` to `100*np.pi` and increase `J` proportionally (the recipe
scales; a single net never will); lower `J` to 5 (patches see several oscillations — spectral
bias creeps back: watch the error plateau); set `OVERLAP = 1.0` (no overlap — the partition
of unity develops seams); compare against the Fourier-features fix from Example 6 on this
problem — two different cures for the same disease.